In [1]:
import sys; sys.path.append('..')
import MeshFEM
import mesh, elastic_sheet, energy, benchmark
import triangulation
from tri_mesh_viewer import TriMeshViewer
import numpy as np

In [2]:
V, E = mesh.load_raw('Data/victorinox.obj')
area = 3.0
V, F, edgeMarkers = triangulation.triangulate(V[:, 0:2], E, triArea=area, outputPointMarkers=False, outputEdgeMarkers=True)
m = mesh.Mesh(V, F)

isBoundary = np.zeros(m.numVertices(), dtype=bool)
isBoundary[m.boundaryVertices()] = True
creases = np.array([em for em in edgeMarkers if not isBoundary[em].all()])

In [3]:
psi = energy.NeoHookeanYoungPoisson(2, 1, 0.3)
es = elastic_sheet.ElasticSheet(m, psi, creases)
es.thickness = 0.05
pinVars, pinVerts = es.prepareRigidMotionPins()
creaseVars = np.arange(es.numCreases()) + es.creaseAngleOffset()

In [4]:
esview = TriMeshViewer(es, wireframe=True, width=1024, height=768)
esview.materialLibrary.material(False).color='#CC1111'
esview.show()

Renderer(camera=PerspectiveCamera(aspect=1.3333333333333333, children=(PointLight(color='#999999', position=(0…

In [5]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.niter = 20

### Rerun the following cell to fold:

In [34]:
es.setCreaseAngles(es.getCreaseAngles()[0] + 0.2 * np.pi / 16 * np.ones(es.numCreases()))
def iter_cb(prob, it):
    return
    if (it % 5 == 1):
       esview.update()
#benchmark.reset()
es.computeEquilibrium(loads=[], fixedVars=pinVars + list(creaseVars), cb=iter_cb, opts=opts)
esview.update()
#benchmark.report()

0	3.79071e-06	4.64463e-06	0.25	0
1	3.67032e-06	3.27365e-05	1	0
2	3.56583e-06	6.0549e-05	1	0
3	3.50542e-06	2.4298e-05	1	0
4	3.49064e-06	1.12458e-05	1	0
5	3.48557e-06	1.64942e-05	1	0
6	3.47975e-06	4.66695e-06	1	0
7	3.47851e-06	3.14416e-06	1	0
8	3.47815e-06	1.08764e-06	1	0
9	3.4781e-06	1.33541e-07	1	0
10	3.4781e-06	1.41141e-08	1	0
